In [2]:
import json

import httpx

from folium import Map, TileLayer


In [6]:
titiler_endpoint = "https://titiler-pgstac-e4drr.replit.app/"
response = httpx.get(f"{titiler_endpoint}/status", headers={"Accept": "application/json"}, follow_redirects=True)

# Check the content before parsing
print(response.status_code)
print(response.headers.get("content-type"))
print(response.text)
print(response.request.url)
print(response.headers)

# Then safely parse JSON
try:
    print(response.json())
except httpx.HTTPError:
    print("Failed to parse JSON")

200
application/json
{"database_online":true,"versions":{"titiler":"0.22.1","titiler.pgstac":"1.8.0","rasterio":"1.4.3","gdal":"3.9.3","proj":"9.4.1","geos":"3.11.1"}}
https://titiler-pgstac-e4drr.replit.app/status
Headers({'access-control-allow-headers': 'DNT,User-Agent,X-Requested-With,If-Modified-Since,Cache-Control,Content-Type,Range,Authorization', 'access-control-allow-methods': 'GET, POST, OPTIONS, PUT, DELETE', 'access-control-allow-origin': '*', 'cache-control': 'public, max-age=3600', 'content-type': 'application/json', 'date': 'Fri, 25 Jul 2025 18:17:18 GMT', 'server': 'Google Frontend', 'strict-transport-security': 'max-age=63072000; includeSubDomains', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'})
{'database_online': True, 'versions': {'titiler': '0.22.1', 'titiler.pgstac': '1.8.0', 'rasterio': '1.4.3', 'gdal': '3.9.3', 'proj': '9.4.1', 'geos': '3.11.1'}}


In [19]:
# Fetch File Metadata to get min/max rescaling values (because the file is stored as float32)
url = "https://opendata.digitalglobe.com/events/mauritius-oil-spill/post-event/2020-08-12/105001001F1B5B00/105001001F1B5B00.tif"
r = httpx.get(
    f"{titiler_endpoint}/cog/info",
    params={
        "url": url,
    },
).json()

bounds = r["bounds"]
print(r)



{'bounds': [57.664053823239804, -20.55473177712791, 57.84021477996238, -20.25261582755764], 'crs': 'http://www.opengis.net/def/crs/EPSG/0/4326', 'band_metadata': [['b1', {}], ['b2', {}], ['b3', {}]], 'band_descriptions': [['b1', ''], ['b2', ''], ['b3', '']], 'dtype': 'uint8', 'nodata_type': 'Mask', 'colorinterp': ['red', 'green', 'blue'], 'scales': [1.0, 1.0, 1.0], 'offsets': [0.0, 0.0, 0.0], 'driver': 'GTiff', 'count': 3, 'width': 38628, 'height': 66247, 'overviews': [2, 4, 8, 16, 32, 64, 128]}


In [20]:
r

{'bounds': [57.664053823239804,
  -20.55473177712791,
  57.84021477996238,
  -20.25261582755764],
 'crs': 'http://www.opengis.net/def/crs/EPSG/0/4326',
 'band_metadata': [['b1', {}], ['b2', {}], ['b3', {}]],
 'band_descriptions': [['b1', ''], ['b2', ''], ['b3', '']],
 'dtype': 'uint8',
 'nodata_type': 'Mask',
 'colorinterp': ['red', 'green', 'blue'],
 'scales': [1.0, 1.0, 1.0],
 'offsets': [0.0, 0.0, 0.0],
 'driver': 'GTiff',
 'count': 3,
 'width': 38628,
 'height': 66247,
 'overviews': [2, 4, 8, 16, 32, 64, 128]}

In [16]:
import json
import httpx
from folium import Map, TileLayer

titiler_endpoint = "https://titiler-pgstac-e4drr.replit.app"
url = "https://opendata.digitalglobe.com/events/mauritius-oil-spill/post-event/2020-08-12/105001001F1B5B00/105001001F1B5B00.tif"

resp = httpx.get(
    f"{titiler_endpoint}/cog/WebMercatorQuad/tilejson.json",
    params={"url": url},
    headers={"Accept": "application/json"},
    follow_redirects=True
)

print("Status code:", resp.status_code)
print("Content-Type:", resp.headers.get("content-type"))
print("First 500 characters of response:")
print(resp.text[:500])

try:
    r = resp.json()
    print("Parsed JSON keys:", r.keys())
except Exception as e:
    print("Failed to parse JSON:", e)

# Fix the tile URL if needed
if r['tiles']:
    # Ensure HTTPS and correct path
    fixed_tiles = []
    for tile_url in r['tiles']:
        tile_url = tile_url.replace('http://', 'https://')
        if '/tiles/' in tile_url and '/cog/tiles/' not in tile_url:
            tile_url = tile_url.replace('/tiles/', '/cog/tiles/')
        fixed_tiles.append(tile_url)
    r['tiles'] = fixed_tiles

# Create the map
#m = Map(location=r['center'][:2], zoom_start=r['center'][2])
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=13
)

# Add the tile layer
tile_layer = TileLayer(
    tiles=r['tiles'][0],
    attr='COG',
    name='COG Layer',
    overlay=True,
    control=True,
    min_zoom=r['minzoom'],
    max_zoom=r['maxzoom']
)
tile_layer.add_to(m)

m

Status code: 200
Content-Type: application/json
First 500 characters of response:
{"tilejson":"2.2.0","version":"1.0.0","scheme":"xyz","tiles":["https://titiler-pgstac-e4drr.replit.app/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?url=https%3A%2F%2Fopendata.digitalglobe.com%2Fevents%2Fmauritius-oil-spill%2Fpost-event%2F2020-08-12%2F105001001F1B5B00%2F105001001F1B5B00.tif"],"minzoom":10,"maxzoom":18,"bounds":[57.664053823239804,-20.55473177712791,57.84021477996238,-20.25261582755764],"center":[57.75213430160109,-20.403673802342773,10]}
Parsed JSON keys: dict_keys(['tilejson', 'version', 'scheme', 'tiles', 'minzoom', 'maxzoom', 'bounds', 'center'])


In [21]:
r = httpx.get(
    f"{titiler_endpoint}/cog/WebMercatorQuad/tilejson.json",
    params={
        "url": url,
    },
).json()
print(r)

{'tilejson': '2.2.0', 'version': '1.0.0', 'scheme': 'xyz', 'tiles': ['https://titiler-pgstac-e4drr.replit.app/tiles/WebMercatorQuad/{z}/{x}/{y}@1x?url=https%3A%2F%2Fopendata.digitalglobe.com%2Fevents%2Fmauritius-oil-spill%2Fpost-event%2F2020-08-12%2F105001001F1B5B00%2F105001001F1B5B00.tif'], 'minzoom': 10, 'maxzoom': 18, 'bounds': [57.664053823239804, -20.55473177712791, 57.84021477996238, -20.25261582755764], 'center': [57.75213430160109, -20.403673802342773, 10]}
